In [11]:
import pandas as pd


In [12]:
read_directory = '../data/results_gathered'
save_directory = '../data/results_gathered/baselines.csv'

In [13]:
filenames = ['mlp', 'mlp_big', 'snn', 'snn_big']

In [21]:
dfs = []

for filename in filenames:
    path = f'{read_directory}/results_gathered_{filename}.csv'
    df = pd.read_csv(path, index_col=0)
    df['baseline'] = filename
    dfs.append(df)

result_df = pd.concat(dfs, ignore_index=True)

In [22]:
result_df = result_df.sort_values(by=["c_index_avg"], ascending=False)

In [23]:
def parse_experiment(exp):
    out = {
        "data_type": "raw",
        "normalization": None,
        "embedding_model": None,
        "hvg": 0
    }
    
    if "scgpt_adata" in exp:
        out["data_type"] = "embedding"
        
        if "pancancer" in exp:
            out["embedding_model"] = "pancancer"
        elif "whole_human" in exp:
            out["embedding_model"] = "whole_human"
        
        if "hvg_" in exp:
            out["hvg"] = int(exp.split("hvg_")[1].split(".")[0])

    if "tpm" in exp:
        out["normalization"] = "TPM"
    elif "counts" in exp:
        out["normalization"] = "counts"
    
    return pd.Series(out)


result_df = result_df.join(result_df["experiment"].apply(parse_experiment))


In [24]:
result_df

,experiment,loss_avg,loss_std,c_index_avg,c_index_std,c_index_ipcw_avg,c_index_ipcw_std,baseline,data_type,normalization,embedding_model,hvg
42,Final_results_TCGA-BRCA.star_tpm.csv,15.844594,4.406341,0.730481,0.083589,0.643454,0.122930,snn_big,raw,TPM,None,0
28,Final_results_TCGA-BRCA.star_tpm.csv,6.124241,1.113554,0.728785,0.080471,0.625501,0.131519,snn,raw,TPM,None,0
43,Final_results_TCGA-BRCA.star_counts.csv,17.282411,3.786621,0.724132,0.066134,0.667841,0.120472,snn_big,raw,counts,None,0
0,Final_results_TCGA-BRCA.star_tpm.csv,3.727684,0.846379,0.722072,0.077890,0.626490,0.112428,mlp,raw,TPM,None,0
1,Final_results_TCGA-BRCA.star_counts.csv,3.595652,0.481881,0.717970,0.049064,0.626261,0.075343,mlp,raw,counts,None,0
29,Final_results_TCGA-BRCA.star_counts.csv,6.570696,0.719155,0.713689,0.041754,0.634991,0.083398,snn,raw,counts,None,0
14,Final_results_TCGA-BRCA.star_counts.csv,4.832736,0.937835,0.712360,0.055748,0.635052,0.094081,mlp_big,raw,counts,None,0
15,Final_results_TCGA-BRCA.star_tpm.csv,4.735203,1.077261,0.708060,0.088001,0.628517,0.120827,mlp_big,raw,TPM,None,0
30,Final_results_scgpt_adata_TCGA-BRCA.star_count...,3.371445,0.695338,0.646102,0.101684,0.537738,0.118326,snn,embedding,counts,pancancer,3000
2,Final_results_scgpt_adata_TCGA-BRCA.star_tpm_p...,3.066455,0.366001,0.645853,0.099764,0.554217,0.056885,mlp,embedding,TPM,pancancer,6000


In [25]:
result_df = result_df[result_df['normalization'] == 'TPM']

In [26]:
result_df.to_csv(save_directory)


In [28]:
raw_vs_emb = (
    result_df.groupby("data_type")
      .agg(
          c_index_mean=("c_index_avg", "mean"),
          c_index_std=("c_index_avg", "std"),
          ipcw_mean=("c_index_ipcw_avg", "mean"),
          ipcw_std=("c_index_ipcw_avg", "std")
      )
)

raw_vs_emb

,c_index_mean,c_index_std,ipcw_mean,ipcw_std
data_type,,,,
embedding,0.594722,0.037766,0.529673,0.026735
raw,0.722350,0.010195,0.630991,0.008403


In [23]:
arch_summary = (
    result_df.groupby("baseline")
      .agg(
          c_index_mean=("c_index_avg", "mean"),
          c_index_std=("c_index_avg", "std")
      )
      .sort_values("c_index_mean", ascending=False)
)

arch_summary

,c_index_mean,c_index_std
baseline,,
snn_big,0.621119,0.057789
snn,0.616184,0.056663
mlp,0.615882,0.054719
mlp_big,0.577550,0.065275


In [33]:
arch_vs_data_summary = (
    result_df
    .groupby(["data_type", "baseline"])
    .agg(
        c_index_mean=("c_index_avg", "mean"),
        # c_index_std=("c_index_avg", "std"),
        icpw_mean=("c_index_ipcw_avg", "mean")
        # icpw_std=("c_index_ipcw_avg", "std")
    )
    .sort_values("c_index_mean", ascending=False)
)

arch_vs_data_summary


c_index_mean  icpw_mean
data_type baseline                         
raw       snn_big       0.730481   0.643454
          snn           0.728785   0.625501
          mlp           0.722072   0.626490
          mlp_big       0.708060   0.628517
embedding snn_big       0.609173   0.542494
          mlp           0.608877   0.529031
          snn           0.603949   0.528471
          mlp_big       0.556886   0.518698

In [26]:
arch_vs_data_summary = (
    result_df
    .groupby(["baseline", "data_type"])
    .agg(
        c_index_mean=("c_index_avg", "mean"),
        c_index_std=("c_index_avg", "std")
    )
    .round(3)
    .unstack("data_type")
)

arch_vs_data_summary


c_index_mean        c_index_std       
data_type    embedding    raw   embedding    raw
baseline                                        
mlp              0.599  0.720       0.035  0.003
mlp_big          0.555  0.710       0.036  0.003
snn              0.599  0.721       0.038  0.011
snn_big          0.603  0.727       0.039  0.004

In [31]:
hvg_summary = (
    result_df[result_df["data_type"] == "embedding"]
    .groupby("hvg")
    .agg(
        c_index_mean=("c_index_avg", "mean"),
        c_index_std=("c_index_avg", "std"),
        ipcw_mean=("c_index_ipcw_avg", "mean"),
        ipcw_std=("c_index_ipcw_avg", "std"),
    )
    .sort_index()
)

hvg_summary

,c_index_mean,c_index_std,ipcw_mean,ipcw_std
hvg,,,,
0,0.588009,0.037535,0.538099,0.025530
3000,0.572128,0.037267,0.508658,0.028245
6000,0.624028,0.016496,0.542263,0.011490


In [34]:
raw_vs_emb = (
    result_df.groupby("embedding_model")
      .agg(
          c_index_mean=("c_index_avg", "mean"),
          c_index_std=("c_index_avg", "std"),
          ipcw_mean=("c_index_ipcw_avg", "mean"),
          ipcw_std=("c_index_ipcw_avg", "std")
      )
)

raw_vs_emb

,c_index_mean,c_index_std,ipcw_mean,ipcw_std
embedding_model,,,,
pancancer,0.588064,0.039528,0.529628,0.028209
whole_human,0.601380,0.036373,0.529719,0.026434
